# Assignment 10: Anomaly detection in air quality sensor data with autoencoders

An autoencoder is trained to compress its input and then reconstruct it. Train one only on
*normal* data and it becomes very good at reconstructing normal data — and noticeably bad at
reconstructing anything else. That reconstruction error turns a generative model into an
anomaly detector, and crucially it needs **no labelled examples of anomalies**.

That property matters enormously for environmental sensor networks. Faults, drift, and
genuine pollution episodes all need catching, and nobody has labelled them in advance. This
is unsupervised learning doing work that supervised learning cannot.

You will use real PM2.5 measurements from the US EPA's Air Quality System — the regulatory
monitoring network for the entire United States.

Answer each numbered question in the empty cell below it.

In [ ]:
import io
import urllib.request
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Input

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_recall_curve, average_precision_score

## Load the data

EPA publishes pre-generated annual files with no API key required. This is daily PM2.5
(parameter 88101) for 2023 — about 10 MB.

In [ ]:
url = "https://aqs.epa.gov/aqsweb/airdata/daily_88101_2023.zip"
with urllib.request.urlopen(url) as resp:
    archive = zipfile.ZipFile(io.BytesIO(resp.read()))

name = archive.namelist()[0]
pm = pd.read_csv(archive.open(name), parse_dates=["Date Local"], low_memory=False)

print(f"{len(pm):,} daily records from {pm['Site Num'].nunique():,} site numbers")
pm[["State Name", "County Name", "Date Local", "Arithmetic Mean", "Latitude", "Longitude"]].head()

Each row is one site-day. `Arithmetic Mean` is the daily mean PM2.5 concentration in
micrograms per cubic metre. A site is identified by the combination of State Code, County
Code, Site Num and POC (the code distinguishing multiple instruments at one location).

In [ ]:
pm["site_id"] = (pm["State Code"].astype(str) + "-" + pm["County Code"].astype(str) + "-"
                 + pm["Site Num"].astype(str) + "-" + pm["POC"].astype(str))

# One row per site per day, pivoted into a site-by-day matrix.
daily = pm.pivot_table(index="Date Local", columns="site_id",
                       values="Arithmetic Mean", aggfunc="mean")
print("site-by-day matrix:", daily.shape)

## Part 1: Explore

1) Plot the network-wide daily mean PM2.5 across 2023. Several large excursions should be
visible — identify the dates of the largest, and find out what caused them. (June 2023 in
the northeastern US is worth looking up.)

2) How complete is the record for each site? Keep only sites reporting on at least 300 days of the year, and report how many survive.

3) Plot the distribution of daily PM2.5 values. Is it symmetric? What transformation might make it easier to model, and why does pollution data usually look like this?

## Part 2: Build the training set

The autoencoder learns what a *normal day across the network* looks like. Each sample will be
one day, described by the readings at a set of sites.

4) Select a region — a single state, or a group of nearby states — with enough well-reporting
sites. Build a matrix of days by sites, fill any remaining gaps sensibly, and report its shape.

5) Standardize the features. Split the year chronologically: train on the first two-thirds, test on the last third. Explain why a random split over days would be inappropriate here.

## Part 3: Train the autoencoder

6) Build a symmetric autoencoder — for example, input dimension down to 16, down to 4, back
up to 16, back to the input dimension — using `Dense` layers. Print the summary.

7) Compile with mean squared error and train, using the input as its own target. Plot the training and validation loss curves.

8) The bottleneck dimension is the key hyperparameter. Try three values and describe the trade-off: what goes wrong when it is too large, and what goes wrong when it is too small?

## Part 4: Detect anomalies

9) Compute the reconstruction error for every day in the test set. Plot it as a time series.

10) Flag days above a threshold of your choosing as anomalies. Justify the threshold — a percentile of the training error distribution is a defensible starting point.

11) Plot the network mean PM2.5 with your flagged days marked. Do the flags land on the episodes you identified in question 1?

12) Examine the reconstruction error per site for one flagged day. Which sites contributed most? Does that localize the event geographically?

## Part 5: Faults or events?

An elevated reconstruction error means *unusual*, not *wrong*. A regional smoke episode and
a broken instrument both look unusual, and only one is a data quality problem.

13) For your flagged days, distinguish between the two. A genuine pollution event should be
spatially coherent — neighbouring sites elevated together — while an instrument fault should
affect one site while its neighbours look normal. Devise a check and apply it.

14) Give one example of each type from your results, with a plot supporting your reasoning.

## Part 6: Baselines and honest evaluation

15) Compare against a simple baseline: a rolling z-score on the network mean. Does the
autoencoder find anything the z-score misses? Be specific about what the extra complexity buys.

16) You have no labels, so you cannot compute precision or recall. Describe how you would
build a labelled evaluation set for this problem if you had a month of analyst time, and what
you would then be able to measure that you currently cannot.

*Write your answer here.*

17) A **variational** autoencoder places a probability distribution over the latent space
rather than mapping each input to a single point. Explain what that buys you for this task —
in particular, what it would let you say about an anomaly score that a plain autoencoder cannot.

*Write your answer here.*

18) Suppose this detector were deployed to automatically flag suspect data for removal from
the regulatory record. Describe one way it could go badly wrong, and what safeguard you would
put in place.

*Write your answer here.*